In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.17 The Hydrogen Atom

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.17",
    title="The Hydrogen Atom",
    blurb="The triumph. Put the Coulomb attraction of a proton for an electron into "
    "the radial equation, and out come the energies that explain the hydrogen "
    "spectral lines, the orbitals that shape chemistry, and a surprise: the energy "
    "depends only on the principal quantum number, not on angular momentum, so the "
    "s, p, and d states of a shell all share one energy. That extra degeneracy is no "
    "accident — it is the signature of a hidden symmetry, the same one that keeps the "
    "planets' orbits from precessing.",
    difficulty="advanced",
    estimate="180–220 min",
)

## Notebook overview

This is the notebook the whole volume has been building toward. Every tool is now in hand — the
eigenvalue solver of [§6.10](schrodinger-on-a-computer.ipynb), the spherical harmonics of [§6.15](orbital-angular-momentum.ipynb), and the radial reduction of [§6.16](central-potentials-3d.ipynb) — and we
point all of them at the one potential that matters most: the **Coulomb attraction** of a proton for an
electron, $V(r)=-Z/r$. What comes out is the **hydrogen atom**, the first real atom ever solved exactly,
and the calculation that turned quantum mechanics from a promising idea into the established theory of
matter.

It delivers two triumphs. The first is the **Rydberg spectrum** $E_n=-13.6\,Z^2/n^2\,$eV: feeding
$V=-Z/r$ into the radial equation of [§6.16](central-potentials-3d.ipynb) yields bound states whose energies depend on a single integer
$n$, and whose *differences* are exactly the spectral lines — Lyman, Balmer, Paschen — that
spectroscopists had catalogued for decades without understanding. Bohr had guessed this formula in 1913
from a model he knew was provisional; here Schrödinger's equation produces it exactly, on a grid, in
seconds. The second triumph is the **orbitals** $\psi_{n,l,m}=R_{n,l}(r)Y_l^m(\theta,\varphi)$ — the
1s, 2s, 2p, 3d shapes that organize all of chemistry, labelled by three quantum numbers: $n$ (the shell),
$l$ (angular momentum, $0\le l\le n-1$), and $m$ (orientation).

And then the surprise. The Coulomb energy depends **only on $n$, not on $l$**: the 2s and 2p states,
though shaped completely differently, have *exactly* the same energy. Rotational symmetry alone would
only make the $2l+1$ orientations degenerate; this extra "accidental" $l$-degeneracy is special to the
$1/r$ potential, and it is no accident at all. It is the fingerprint of a **hidden symmetry** — the
conservation of the quantum **Runge–Lenz vector**, which enlarges the rotation group $SO(3)$ to $SO(4)$,
the very same conserved quantity that keeps a classical Kepler orbit from precessing (the thread from
Volumes I, II, and IV, where Mercury's *failure* to close revealed relativity). Counting a shell's states
gives $n^2$ orbitals, $2n^2$ with spin — exactly the $2,8,18,32$ capacities of the periodic table's rows.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the [§6.16](central-potentials-3d.ipynb) radial reduction $u=rR$ with the Coulomb $V_{\text{eff}}
=-Z/r+l(l+1)/2r^2$, the efficient `scipy.linalg.eigh_tridiagonal` for the radial eigenproblem, and
`scipy.special.genlaguerre` for the analytic hydrogen radial functions in the comparison.

> **Units and method notes.** We work in **atomic units** ($\hbar=m_e=e=4\pi\varepsilon_0=1$), so
> energies are in **Hartree** ($1\,\text{Ha}=27.211\,$eV) and lengths in **Bohr radii** ($a_0=1$, i.e.
> $0.529\,$Å). The radial Hamiltonian is **tridiagonal** (the $(1,-2,1)$ stencil), so we use
> `scipy.linalg.eigh_tridiagonal` rather than the dense `numpy.linalg.eigh` of [§6.10](schrodinger-on-a-computer.ipynb) — the dense solver
> is far too slow on the fine grids and many $l$ values this notebook needs. **Numerical honesty:** high-
> $n$ orbitals are large and diffuse ($\langle r\rangle\sim n^2 a_0$), so the box $r_{\max}$ must be big
> enough to contain them, or their energies come out spuriously *high* — a box artifact, not physics
> (the [§6.10](schrodinger-on-a-computer.ipynb)/[§6.11](bound-states-1d.ipynb) lesson). We keep comparisons to well-contained levels. The radial node count is
> $n_r=n-l-1$. See Sakurai & Napolitano and Griffiths (the hydrogen atom, the degeneracy, the Runge–Lenz
> vector); and Notebooks [§6.16](central-potentials-3d.ipynb) (the radial equation), [§6.15](orbital-angular-momentum.ipynb) (the spherical harmonics), [§6.10](schrodinger-on-a-computer.ipynb) (the
> eigenmethod), [§6.12](harmonic-oscillator.ipynb) (Laguerre/Hermite special functions).

## Theory in brief

### The Coulomb problem

The electron in a hydrogen-like atom feels the central potential

```{math}
:label: eq-hydrogen-coulomb
V(r)=-\frac{Ze^2}{4\pi\varepsilon_0 r}=-\frac{Z}{r}\quad(\text{atomic units}),\qquad V_{\text{eff}}(r)=-\frac{Z}{r}+\frac{l(l+1)}{2r^2} .
```

This is a central potential ([§6.16](central-potentials-3d.ipynb)), so the stationary states factor as $\psi_{n,l,m}=R_{n,l}(r)Y_l^m(
\theta,\varphi)$ and the radial function obeys the 1-D radial equation with the Coulomb well plus the
centrifugal barrier. Their competition — the $-Z/r$ attraction versus the $l(l+1)/r^2$ repulsion — sets
the sizes of the orbitals.

### The Rydberg spectrum

Solving the radial equation (a power-series analysis that Griffiths carries out in full) gives the
bound-state energies

```{math}
:label: eq-rydberg
E_n=-\frac{Z^2}{2n^2}\ \text{Hartree}=-\frac{13.6\,Z^2}{n^2}\ \text{eV},\qquad n=n_r+l+1=1,2,3,\dots ,
```

with $n$ the **principal** quantum number and $n_r$ the number of radial nodes. The *differences*
$E_n-E_{n'}$ are the spectral-line energies: the **Lyman** series (to $n=1$, ultraviolet), **Balmer** (to
$n=2$, visible), and **Paschen** (to $n=3$, infrared). This is the formula that explained the hydrogen
spectrum — quantum mechanics' first great empirical triumph.

### The three quantum numbers and the orbitals

The separation of variables in [§6.16](central-potentials-3d.ipynb) did most of the labelling for us: the angular equation fixed
$Y_l^m$ with its integers $l$ and $m$, and the radial equation contributes one more integer, the node
count $n_r$, which combines with $l$ into the principal quantum number $n$. Every bound state therefore
carries three labels:

```{math}
:label: eq-orbitals
\psi_{n,l,m}(r,\theta,\varphi)=R_{n,l}(r)\,Y_l^m(\theta,\varphi),\qquad 0\le l\le n-1,\quad -l\le m\le l,\quad n_r=n-l-1 .
```

The radial function $R_{n,l}$ (an exponential times an associated Laguerre polynomial) has $n_r=n-l-1$
nodes; the angular factor is a spherical harmonic ([§6.15](orbital-angular-momentum.ipynb)). Together they are the **atomic orbitals** —
1s, 2s, 2p, 3s, 3p, 3d, and so on. The radial density $r^2|R|^2$ gives the probability of finding the
electron at radius $r$; for the ground state it peaks at the **Bohr radius** $a_0$.

### The $l$-degeneracy and the hidden symmetry

In a generic central potential the energy depends on $n_r$ and $l$ separately, as the wells of [§6.16](central-potentials-3d.ipynb)
showed. For the Coulomb potential something remarkable happens: the energies {eq}`eq-rydberg` collapse
onto the single combination $n=n_r+l+1$, so that

```{math}
:label: eq-degeneracy
E_n\ \text{depends on } n\ \text{only, not on } l\ \Longrightarrow\ \text{all } l=0,\dots,n-1\ \text{share the energy } -Z^2/2n^2 .
```

Rotational symmetry alone guarantees only the $(2l+1)$-fold $m$-degeneracy; the extra $l$-degeneracy is
"accidental" — the fingerprint of a **hidden symmetry** special to $1/r$. The quantum **Runge–Lenz
vector** $\mathbf A$ is conserved (as it is classically, where it points along the fixed major axis of a
non-precessing Kepler ellipse), and together with $\mathbf L$ it generates the group $SO(4)$, larger than
$SO(3)$. This enlarged symmetry forces the $l$-degeneracy — the same phenomenon as the isotropic
oscillator's $2n_r+l$ degeneracy ([§6.16](central-potentials-3d.ipynb)), here in its sharpest form. Any departure from exact $1/r$ — the
screening of the nucleus by inner electrons in a many-electron atom — **breaks** the symmetry and splits
$s,p,d$, and *that splitting is why the periodic table has the structure it does*.

### Shell degeneracy and the periodic table

Counting the states of shell $n$ is now pure arithmetic: each allowed $l$ contributes its $2l+1$
orientations, and the $l$-degeneracy just established puts them all at one energy, so the sum runs over
$l=0,\dots,n-1$ and gives

```{math}
:label: eq-shells
\sum_{l=0}^{n-1}(2l+1)=n^2\ \text{orbitals per shell}\ \Longrightarrow\ 2n^2\ \text{with spin}=2,8,18,32,\dots
```

the exact row capacities of the periodic table. The shell structure of matter is the degeneracy counting
of the hydrogen atom (spin, the factor of 2, is added properly in [§6.18](spin-magnetic.ipynb)).

### Realism caveats

Before trusting the triumph, one should ask what the Hamiltonian left out. The answer, quantified by
the perturbative estimates that Griffiths works through systematically, is: nothing above the $10^{-3}$
level in relative terms, because every neglected effect is suppressed by powers of the fine-structure
constant $\alpha\approx1/137$ or of the mass ratio $m_e/m_p$. In summary,

```{math}
:label: eq-caveats
E_1=-13.6\,\text{eV}\ \text{is the gross structure; the corrections are}\ \lesssim10^{-3}\ \text{of it.}
```

This is the non-relativistic, infinite-nuclear-mass, spinless-Coulomb atom. Real hydrogen has small
corrections — the finite-mass (reduced-mass) shift, **fine structure** (relativistic + spin–orbit, [§6.21](perturbation-fine-structure.ipynb)),
the **Lamb shift** (QED), and **hyperfine** structure (the 21-cm line) — all tiny on the scale of
$-13.6\,$eV. The gross structure computed here is right to about $0.05\%$ (reduced-mass-limited).

## Setup

The data are the series palette, the atomic-unit conversions ($1\,$Ha in eV, the Bohr radius in
ångströms, $hc$ in eV·nm), and the **analytic** hydrogen radial functions — the closed Laguerre
form of {eq}`eq-orbitals`, which is the reference standard the grid solutions are checked
against, not something to construct. The instruments are the radial density, which is nothing
but $|u|^2$ once $u=rR$, and the **real** spherical harmonics, built from scratch in
[§3.5](../03-electrodynamics/multipole-expansion.ipynb) and restated here as a tool for the
orbital cross-sections. The Coulomb radial solver is deliberately absent: you build
`solve_hydrogen_radial` in Exercise 1, and Exercises 2, 4, 5, 6 and 8 all run on the one you
wrote.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eigh_tridiagonal
from scipy.special import factorial, genlaguerre

from ecp import draw, validate

# data: the series palette
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# data: atomic units (ℏ = m_e = e = 4πε₀ = 1) and the conversions out of them
HARTREE_EV = 27.211386  # 1 Hartree in electron-volts
BOHR_ANGSTROM = 0.529177  # the Bohr radius a₀ in ångströms
HC_EV_NM = 1239.841984  # hc in eV·nm (for wavelengths)


# data: the exact answer, handed over so the grid can be judged against it. This is the closed
# form quoted in the theory section, transcribed; the orbitals this notebook *computes* come out
# of the solver you write in Exercise 1, and this is only what they are measured against.
def hydrogen_radial_analytic(n, l, r, Z=1):
    r"""The exact hydrogen radial function $R_{n,l}(r)$, via ``scipy.special.genlaguerre``.

    $R_{n,l}(r)=N_{nl}\,e^{-Zr/n}(2Zr/n)^l L_{n-l-1}^{2l+1}(2Zr/n)$ with $L$ the associated Laguerre
    polynomial and $N_{nl}$ the normalization $\int|R|^2r^2dr=1$. Used to check the grid solutions
    {eq}`eq-orbitals`.
    """
    norm = np.sqrt((2 * Z / n) ** 3 * factorial(n - l - 1) / (2 * n * factorial(n + l)))
    rho = 2 * Z * r / n
    return norm * np.exp(-rho / 2) * rho**l * genlaguerre(n - l - 1, 2 * l + 1)(rho)


# instrument: because u = rR, the radial density IS |u|² — there is nothing to derive and nothing
# to choose. Naming the absolute square keeps the peak-finding of Exercises 5 and 8 readable; the
# lesson there is *where* the density peaks, not the squaring that forms it.
def radial_density(u):
    r"""The radial probability density $|u(r)|^2=r^2|R(r)|^2$ — the probability per unit $r$ (§6.16)."""
    return np.abs(u) ** 2


# built from scratch in §3.5 (Exercise 6, where the cosine/sine combinations are assembled and
# drawn as the orbital lobes); restated here as an instrument for the orbital cross-sections.
def real_spherical_harmonic(l, m, theta, phi):
    r"""The **real** spherical harmonic (the orbital angular shape), reused from §6.15 / Volume III."""
    from scipy.special import sph_harm_y

    if m == 0:
        return sph_harm_y(l, 0, theta, phi).real
    if m > 0:
        return np.sqrt(2.0) * (-1.0) ** m * sph_harm_y(l, m, theta, phi).real
    return np.sqrt(2.0) * (-1.0) ** m * sph_harm_y(l, -m, theta, phi).imag

## Exercise 1 — The Coulomb radial equation and the ground state

Every tool of the volume now meets one potential. The [§6.16](central-potentials-3d.ipynb) reduction $u=rR$ has already turned
the three-dimensional problem into a one-dimensional radial equation, and dropping the Coulomb
attraction into it gives $V_{\text{eff}}(r)=-Z/r+l(l+1)/2r^2$ {eq}`eq-hydrogen-coulomb` — for $s$
states ($l=0$) the bare well $-Z/r$, with no centrifugal barrier at all. Discretized on an
interior grid $r\in(0,r_{\max})$ that excludes both endpoints (so $u(0)=u(r_{\max})=0$ by
construction), the radial Hamiltonian is **tridiagonal**: the $(1,-2,1)/dr^2$ kinetic stencil plus
$\mathrm{diag}\,V_{\text{eff}}$ has a diagonal $1/dr^2+V_{\text{eff}}$ and a single off-diagonal
$-\tfrac12/dr^2$, and nothing else. That is why `scipy.linalg.eigh_tridiagonal` replaces the dense
`numpy.linalg.eigh` of [§6.10](schrodinger-on-a-computer.ipynb) here: it needs only those two vectors, and the fine grids and many
$l$ values the rest of this notebook asks for would be far too slow dense. Dividing the
eigenvectors by $\sqrt{dr}$ normalizes them to $\int|u|^2dr=1$. The number the whole construction
must produce is the one Bohr guessed in 1913: $E_1=-0.5\,$Ha $=-13.6\,$eV {eq}`eq-rydberg`, the
ground state of the first atom ever solved exactly.

1. Write `solve_hydrogen_radial(l, Z, rmax, N)`, returning the interior grid `r`, the ascending
   radial energies, and the matrix whose column $n_r$ is $u_{n_r,l}=rR$: build the grid with
   `numpy.linspace`, form $V_{\text{eff}}$, assemble the tridiagonal Hamiltonian's diagonal and
   off-diagonal, diagonalize with `scipy.linalg.eigh_tridiagonal`, and divide by $\sqrt{dr}$.
   **Write this one yourself** — the implementation is the lesson.
2. Solve it at $l=0$, $Z=1$ on a box wide enough to hold the ground state, and read the lowest
   energy.
3. Confirm $E_1=-0.5\,$Ha and convert to eV with `HARTREE_EV`.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    E1,
    -0.5,
    "the hydrogen ground state is E₁ = −0.5 Ha = −13.6 eV (the first exactly-solved atom)",
    atol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The Rydberg spectrum

One value of $l$ gives one ladder; the spectrum is what happens when the ladders are laid side by
side. Each fixed-$l$ solve returns its bound states ($E<0$) in order, so the level at index $n_r$
is the one with $n_r$ radial nodes, and its **principal** quantum number is $n=n_r+l+1$
{eq}`eq-orbitals`. Labelled that way, every level should fall on $E_n=-1/2n^2\,$Ha
$=-13.6/n^2\,$eV {eq}`eq-rydberg` — the formula that explained the spectral lines, and the reason
this calculation mattered. One caution before reading the numbers: a level only lands on its
Rydberg value if the box holds it, and since $\langle r\rangle\sim n^2a_0$ the high-$n$ states
outgrow any fixed $r_{\max}$ and come out spuriously *high*. That is a box artifact, not physics
(the [§6.10](schrodinger-on-a-computer.ipynb)/[§6.11](bound-states-1d.ipynb) lesson), so the comparison below is kept to the well-contained levels.

1. With the `solve_hydrogen_radial` you wrote in Exercise 1, solve for $l=0,1,2$ and collect the
   bound energies.
2. Assign each level its principal number $n=n_r+l+1$.
3. Compare the well-contained levels to $-1/2n^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    np.array(E_grid),
    np.array(E_exact),
    "the hydrogen spectrum is Eₙ = −Z²/2n² Hartree (the Rydberg formula), for every (n,l)",
    atol=2e-3,
)

```{admonition} With your assistant
:class: tip
Ask your assistant for a radial-grid convergence sweep of its own design —
grid sizes, box radii, however it structures the loop — and run it. The gate
is the one this atom is famous for: the computed levels must approach
$E_n = -1/(2n^2)$ Hartree, and a sweep that "converges" anywhere else has
converged to its own discretization, not to hydrogen. The check is yours.
```

## Exercise 3 — Spectral lines: Lyman, Balmer, Paschen

What a spectroscope records is not an energy level but a *difference* between two of them: an
electron falling from $n'$ to $n$ emits a photon carrying $\Delta E=E_{n'}-E_n$, which appears at
the wavelength $\lambda=hc/\Delta E$ (with $hc=1239.84\,$eV·nm, the `HC_EV_NM` constant). Grouping
the transitions by their lower level sorts them into the classic series: **Lyman** down to $n=1$
(ultraviolet), **Balmer** down to $n=2$ (visible), **Paschen** down to $n=3$ (infrared). The
Balmer lines are the ones a person can actually see — $H\alpha$ ($3\to2$) observed at $656.3\,$nm,
$H\beta$ ($4\to2$) at $486\,$nm — and Balmer fit them by eye in 1885, four decades before there
was a theory to fit. Reproducing that catalogue line for line, out of {eq}`eq-rydberg` alone, is
quantum mechanics' most direct empirical triumph.

1. From $E_n=-1/2n^2\,$Ha, form the transition energies $\Delta E=E_{n'}-E_n$ for $n'>n$.
2. Convert them to wavelengths $\lambda=hc/\Delta E$.
3. Compare the Balmer $H\alpha$ and $H\beta$ wavelengths to the observed lines.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    h_alpha,
    656.3,
    "the energy differences reproduce the hydrogen spectral series — the Balmer Hα line at 656 nm",
    atol=2.0,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — The orbitals: radial functions and shapes

The energies are only half of what the radial equation returns. The eigenvectors are the reduced
functions $u_{n_r,l}$, and dividing out the reduction, $R_{n,l}=u/r$, gives the radial factor of
the orbital $\psi_{n,l,m}=R_{n,l}Y_l^m$ {eq}`eq-orbitals`. There is an exact answer to check them
against: $R_{n,l}$ is an exponential times an associated Laguerre polynomial, the closed form
`hydrogen_radial_analytic` supplies from `scipy.special.genlaguerre`, and the two should agree to
the discretization error. The structural fingerprint to look for is the node count $n_r=n-l-1$ —
the 1s and 2p are nodeless, the 2s changes sign once — which is exactly the index at which each
state sits in its fixed-$l$ list. Two cautions on the comparison: a diagonalizer returns
eigenvectors up to an arbitrary overall sign, so one of the two curves may need flipping before
they can be subtracted, and the agreement should be judged where the function has appreciable
amplitude, not out in the exponential tail where both are numerically zero. Wrapping the radial
factor in the [§6.15](orbital-angular-momentum.ipynb) angular shapes then gives the full orbital density $|\psi_{n,l,m}|^2$ — the
1s, 2s, 2p, 3d pictures that organize chemistry.

1. With the `solve_hydrogen_radial` you wrote in Exercise 1, extract $R_{n,l}(r)=u/r$ from the
   grid for $(1,0),(2,0),(2,1)$.
2. Compare each to the analytic Laguerre form and confirm the radial node count $n_r=n-l-1$
   (1s: 0, 2s: 1, 2p: 0).
3. Plot $R_{n,l}$, the radial densities $r^2|R|^2$, and a cross-section of the full orbital
   density $|\psi_{n,l,m}|^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    match_ok,
    "the grid radial functions match the analytic hydrogen orbitals (scipy.special.genlaguerre) with n_r=n−l−1 nodes",
)

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The most probable radius and the Bohr radius

"How big is a hydrogen atom" has a sharp answer once the wavefunction is in hand, but it needs the
right question. The quantity to look at is the radial probability density $r^2|R_{10}|^2$, which
because $u=rR$ is simply $|u_{10}|^2$ — the probability of finding the electron in a thin shell at
radius $r$, angles integrated out. Its maximum is the **most probable** radius, and for the ground
state it should land on the **Bohr radius** $a_0$, which in atomic units is exactly $1$
{eq}`eq-orbitals`: the length Bohr postulated in 1913, here recovered from the wavefunction rather
than assumed. It is worth contrasting that with the *mean* radius $\langle r\rangle=\int r|u|^2dr$,
which comes out at $1.5\,a_0$ — larger, because the density has a long outward tail that pulls the
average out while leaving the peak where it is. Note also that $R$ itself is largest at the
nucleus; it is the $r^2$ shell volume that moves the most likely radius outward.

1. With the `solve_hydrogen_radial` you wrote in Exercise 1, form the ground-state radial density
   $|u_{10}|^2$ (`radial_density`).
2. Find its maximum (`numpy.argmax`) and confirm the peak is at $r=a_0$.
3. Compute the mean radius $\langle r\rangle$ and compare it with the peak.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    r_peak,
    1.0,
    "the ground-state radial density peaks at the Bohr radius a₀ (=1 atomic unit)",
    atol=2e-2,
)

## Exercise 6 — The $l$-degeneracy and the hidden symmetry

In a generic central potential the energy depends on $n_r$ and $l$ separately — the wells of
[§6.16](central-potentials-3d.ipynb) showed exactly that. The Coulomb potential is not generic: its levels collapse onto the
single combination $n=n_r+l+1$, so the 2s and 2p, shaped nothing alike, sit at *precisely* the same
energy {eq}`eq-degeneracy`. Rotational symmetry cannot account for this. Invariance under
rotations guarantees only that the $2l+1$ orientations of a given $l$ are degenerate; that
different $l$ agree is an extra coincidence, called "accidental" for as long as no one could say
what caused it. What causes it is a **hidden symmetry** special to $1/r$: the quantum
**Runge–Lenz vector** $\mathbf A$ is conserved, and together with $\mathbf L$ it generates $SO(4)$
rather than $SO(3)$, an enlarged symmetry that forces levels of different $l$ together. It is the
same conserved vector that fixes the major axis of a classical Kepler ellipse and keeps it from
precessing (Volumes I, II, and IV). The converse matters just as much: any departure from exact
$1/r$ — inner electrons screening the nucleus in a many-electron atom — breaks the symmetry and
splits $s$ from $p$ from $d$, and that splitting is what gives the periodic table its structure.
Note that the levels below are three *independent* diagonalizations, one per $l$, agreeing to five
decimal places; nothing in the code arranges for that.

1. With the `solve_hydrogen_radial` you wrote in Exercise 1, collect the energies of the $(n,l)$
   states for $n=1..4$, $l=0..n-1$ (the lowest state of each $l$ has $n=l+1$).
2. Confirm all $l$ at fixed $n$ share the energy $-1/2n^2$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    np.array(deg_grid),
    np.array(deg_exact),
    "hydrogen's energy depends only on n, not l — the SO(4) hidden-symmetry (Runge–Lenz) degeneracy",
    atol=2e-3,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Shell degeneracy and the periodic table *(student)*

With the $l$-degeneracy of Exercise 6 established, counting the states of a shell is pure
arithmetic. Each allowed $l$ contributes its $2l+1$ orientations, the degeneracy puts them all at
one energy, and the sum runs over $l=0,\dots,n-1$ to give $n^2$ orbitals per shell, or $2n^2$ once
the electron's two spin states are counted ([§6.18](spin-magnetic.ipynb)) {eq}`eq-shells`. Those numbers —
$2,8,18,32$ — are the row capacities of the periodic table. The shell structure of matter is
hydrogen's degeneracy counting, with one caveat worth stating plainly: in real atoms screening
reorders the subshells (4s fills before 3d), so the *filling order* is not hydrogen's, even though
the counting is.

1. For $n=1..4$, count the orbitals $\sum_{l=0}^{n-1}(2l+1)$ and check the sum equals $n^2$.
2. Multiply by 2 for spin and compare to the periodic table's row lengths $2,8,18,32$.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    counts_ok,
    "shell n holds n² orbitals (2n² with spin) = the periodic-table row capacities 2,8,18,32 — the n² degeneracy underlies the periodic table",
)

## Exercise 8 — Hydrogen-like ions and scaling *(student)*

The nuclear charge $Z$ was carried along all through {eq}`eq-hydrogen-coulomb` and
{eq}`eq-rydberg` without ever being set to anything but 1, and collecting it now buys a whole
isoelectronic family — He$^+$, Li$^{2+}$, and every other one-electron ion — from the same
solution. A stronger nucleus binds more tightly and pulls the electron in: energies deepen as
$E_n=-Z^2/2n^2$, and the orbitals shrink as $1/Z$, so the ground-state density peaks at $a_0/Z$.
One practical consequence of that shrinking is that the box should shrink with it — scaling
$r_{\max}$ down by $Z$ keeps each ion's grid resolving its own compressed wavefunction instead of
spending points on empty space.

1. With the `solve_hydrogen_radial` you wrote in Exercise 1, solve at nuclear charge $Z=1,2,3$.
2. Confirm the energies scale as $E_n=-Z^2/2n^2$.
3. Confirm the ground-state density peak moves to $a_0/Z$.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    np.array(scale_grid),
    np.array(scale_exact),
    "hydrogen-like ion energies scale as Eₙ = −Z²/2n² and orbitals shrink as 1/Z",
    rtol=1e-2,
)

## Exercise 9 — The atom that built the theory *(synthesis)*

Everything in this volume came together here. The spherical harmonics gave us the angles ([§6.15](orbital-angular-momentum.ipynb)), the
radial reduction gave us the distance ([§6.16](central-potentials-3d.ipynb)), and the eigenvalue solver gave us the energies ([§6.10](schrodinger-on-a-computer.ipynb)) — and
out of the three came the hydrogen atom: the Rydberg spectrum that matched the spectral lines to a
fraction of a percent, the orbitals that shape every molecule, and a degeneracy so exact it revealed a
symmetry no one had put in by hand.

There is no new computation to do here; the atom is the result. The energy's blindness to $l$
is the quantum echo of the Kepler ellipse that never precesses — the conserved Runge–Lenz vector, the
same $SO(4)$ symmetry we first met as the *closed orbit* of an inverse-square force in Volumes I and II,
and whose tiny violation (Mercury's precession) opened the door to general relativity in Volume IV. Here
it is exact, and its *breaking* — by letting inner electrons screen the nucleus, so the potential is no
longer quite $1/r$ — is what splits $s$ from $p$ from $d$ and gives the periodic table its rows and
columns. What remains is to add the one thing the Schrödinger equation left out: the electron's **spin**,
the fourth quantum number, which the next notebook ([§6.18](spin-magnetic.ipynb)) restores and which completes the $2n^2$ counting
— and which fine structure ([§6.21](perturbation-fine-structure.ipynb)) will use to split these perfect Coulomb levels at last.

Bohr guessed the $-13.6\,$eV$/n^2$ formula in 1913 from a model he knew was provisional. Thirteen years
later the Schrödinger equation produced it *exactly*, along with the orbitals Bohr could not have drawn,
and hid inside it a symmetry that took another decade to name. We computed the whole of it, on a grid, in
an afternoon.

## Notebook summary

The hydrogen atom — the crown of Movement III, and a summit of the volume.

- **The Coulomb problem** {eq}`eq-hydrogen-coulomb`: $V=-Z/r$ in the [§6.16](central-potentials-3d.ipynb) radial equation, $V_{\text{eff}}=-Z/r+
  l(l+1)/2r^2$, solved with `scipy.linalg.eigh_tridiagonal` in atomic units.
- **The Rydberg spectrum** {eq}`eq-rydberg`: $E_n=-Z^2/2n^2\,$Ha $=-13.6\,Z^2/n^2\,$eV; the differences
  are the Lyman/Balmer/Paschen lines ($H\alpha=656\,$nm), matched to spectroscopy.
- **The orbitals** {eq}`eq-orbitals`: $\psi_{n,l,m}=R_{n,l}Y_l^m$, radial nodes $n_r=n-l-1$, matching the
  analytic Laguerre forms to $\sim10^{-4}$; the 1s density peaks at the Bohr radius $a_0$.
- **The $l$-degeneracy** {eq}`eq-degeneracy`: $E$ depends on $n$ only — the Runge–Lenz / $SO(4)$ hidden
  symmetry (the closed Kepler orbit), broken by screening in many-electron atoms.
- **Shell degeneracy** {eq}`eq-shells`: $n^2$ orbitals ($2n^2$ with spin) $=2,8,18,32$ — the periodic
  table's rows.
- **Scaling**: hydrogen-like ions have $E\propto Z^2$ and sizes $\propto1/Z$.

The first exact atom, the spectrum that explained the lines, and a hidden symmetry linking the quantum
atom to the classical Kepler orbit. Only spin is missing — and that is next.

## Outlook

- **Spin ([§6.18](spin-magnetic.ipynb))**: the fourth quantum number $m_s$, the electron's intrinsic angular momentum (the
  half-integers of [§6.14](angular-momentum-algebra.ipynb)), completing the $2n^2$ counting.
- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))**: relativistic and spin–orbit corrections lift the $l$-degeneracy; the Lamb
  shift (QED) and hyperfine structure (the 21-cm line) are further, tinier splittings (horizons).
- **Many-electron atoms and the periodic table** (a horizon): screening breaks the $SO(4)$ symmetry and
  orders the subshells, with the exclusion principle filling them.
- **The Runge–Lenz vector and $SO(4)$** (an algebraic horizon): hydrogen solved by symmetry alone, with no
  differential equation — the Pauli/Fock method.
- **Cross-reference** [§6.16](central-potentials-3d.ipynb) (the radial equation), [§6.15](orbital-angular-momentum.ipynb) (the spherical harmonics), [§6.10](schrodinger-on-a-computer.ipynb) (the eigenmethod),
  [§6.12](harmonic-oscillator.ipynb) (Laguerre/Hermite special functions), and forward to [§6.18](spin-magnetic.ipynb), [§6.21](perturbation-fine-structure.ipynb).

In [ ]:
from ecp.style import footer

footer()